In [20]:
import pandas as pd
# load the extended data
extension = pd.read_csv('../data/transformed_candy_3.csv')
candy = pd.read_csv('../data/candy-data-cleaned.csv')
extension.drop(columns=['usda_number', 'candy_detail', 'NDB_No'], inplace=True)
extension.rename(columns={'candy_names': 'competitorname'}, inplace=True)

In [21]:
names = candy.competitorname.tolist()
from utils import find_closest_match

matched_names = [find_closest_match(name, extension['competitorname'].tolist())[0] for name in names]

In [22]:
candy['alt_names'] = matched_names

In [23]:
candy[['competitorname', 'alt_names']]

,competitorname,alt_names
0,100 Grand,100 Grand
1,3 Musketeers,3 Musketeers
2,One dime,None
3,One quarter,None
4,Air Heads,None
...,...,...
80,Twizzlers,Twizzlers
81,Warheads,None
82,Welch's Fruit Snacks,FRUIT SNACKS
83,Werther's Original Caramel,None


In [24]:
candy

,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent,mother_company,big_brand,alt_names
0,100 Grand,1,0,1,0,0,1,0,1,0,0.901261,0.860,0.670,Ferrero SpA,1,100 Grand
1,3 Musketeers,1,0,0,0,1,0,0,1,0,0.445922,0.511,0.676,"Mars, Inc.",1,3 Musketeers
2,One dime,0,0,0,0,0,0,0,0,0,-1.663576,0.116,0.323,Unknown,0,None
3,One quarter,0,0,0,0,0,0,0,0,0,-1.663576,0.511,0.461,Unknown,0,None
4,Air Heads,0,1,0,0,0,0,0,0,0,1.520236,0.511,0.523,Perfetti Van Melle,0,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,Twizzlers,0,1,0,0,0,0,0,0,0,-0.920093,0.116,0.455,Hershey's,1,Twizzlers
81,Warheads,0,1,0,0,0,0,1,0,0,-1.371875,0.116,0.390,Impact Confections,0,None
82,Welch's Fruit Snacks,0,1,0,0,0,0,0,0,1,-0.589262,0.313,0.444,"The Promotion In Motion Companies, Inc.",0,FRUIT SNACKS
83,Werther's Original Caramel,0,0,1,0,0,0,1,0,0,-1.041043,0.267,0.419,August Storck KG,0,None


In [25]:
candy = candy.join(extension.set_index('competitorname'), on='alt_names', how='left', rsuffix='_ext').drop(columns=['alt_names'])

In [28]:
# input missing values with mean of the column
for col in candy.columns:
    if candy[col].isnull().sum() > 0:
        mean_value = candy[col].mean()
        candy[col].fillna(mean_value, inplace=True)

/var/folders/3r/pv_sqnpd6bb8wr1nhrml1snw0000gq/T/ipykernel_15364/2988800726.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  candy[col].fillna(mean_value, inplace=True)


In [30]:
candy

,competitorname,chocolate,fruity,caramel,peanutyalmondy,nougat,crispedricewafer,hard,bar,pluribus,sugarpercent,pricepercent,winpercent,mother_company,big_brand,Water_(g),Energ_Kcal,Protein_(g),Sugar_Tot_(g)
0,100 Grand,1,0,1,0,0,1,0,1,0,0.901261,0.860,0.670,Ferrero SpA,1,6.100000,468.000000,2.500000,51.900000
1,3 Musketeers,1,0,0,0,1,0,0,1,0,0.445922,0.511,0.676,"Mars, Inc.",1,5.800000,436.000000,2.600000,66.890000
2,One dime,0,0,0,0,0,0,0,0,0,-1.663576,0.116,0.323,Unknown,0,5.107907,461.953488,5.346977,54.558837
3,One quarter,0,0,0,0,0,0,0,0,0,-1.663576,0.511,0.461,Unknown,0,5.107907,461.953488,5.346977,54.558837
4,Air Heads,0,1,0,0,0,0,0,0,0,1.520236,0.511,0.523,Perfetti Van Melle,0,5.107907,461.953488,5.346977,54.558837
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80,Twizzlers,0,1,0,0,0,0,0,0,0,-0.920093,0.116,0.455,Hershey's,1,15.000000,348.000000,2.560000,39.640000
81,Warheads,0,1,0,0,0,0,1,0,0,-1.371875,0.116,0.390,Impact Confections,0,5.107907,461.953488,5.346977,54.558837
82,Welch's Fruit Snacks,0,1,0,0,0,0,0,0,1,-0.589262,0.313,0.444,"The Promotion In Motion Companies, Inc.",0,11.840000,352.000000,0.080000,68.180000
83,Werther's Original Caramel,0,0,1,0,0,0,1,0,0,-1.041043,0.267,0.419,August Storck KG,0,5.107907,461.953488,5.346977,54.558837


In [31]:
import statsmodels.api as sm
features = ['chocolate', 'peanutyalmondy', 'pricepercent','Water_(g)', 'Energ_Kcal', 'Protein_(g)', 'Sugar_Tot_(g)']
target = 'winpercent'
X = candy[features]
y = candy[target]
X = sm.add_constant(X)  # Adds a constant term to the predictors
model = sm.OLS(y, X).fit()
model.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:             winpercent   R-squared:                       0.432
Model:                            OLS   Adj. R-squared:                  0.381
Method:                 Least Squares   F-statistic:                     8.478
Date:                Wed, 05 Nov 2025   Prob (F-statistic):           1.11e-07
Time:                        21:36:47   Log-Likelihood:                 67.441
No. Observations:                  86   AIC:                            -118.9
Df Residuals:                      78   BIC:                            -99.25
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
==================================================================================
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const              0.1432      0.462      0.310      0.758      -0.777       1.063
chocolate          0.1539      0.032      4.769      0.000       0.090       0.218
peanutyalmondy     0.0836      0.045      1.863      0.066      -0.006       0.173
pricepercent      -0.0168      0.056     -0.301      0.765      -0.128       0.094
Water_(g)         -0.0040      0.010     -0.409      0.684      -0.023       0.015
Energ_Kcal         0.0005      0.001      0.645      0.521      -0.001       0.002
Protein_(g)       -0.0034      0.010     -0.350      0.727      -0.023       0.016
Sugar_Tot_(g)      0.0020      0.003      0.657      0.513      -0.004       0.008
==============================================================================
Omnibus:                        0.103   Durbin-Watson:                   1.703
Prob(Omnibus):                  0.950   Jarque-Bera (JB):                0.283
Skew:                           0.027   Prob(JB):                        0.868
Kurtosis:                       2.724   Cond. No.                     1.73e+04
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 1.73e+04. This might indicate that there are
strong multicollinearity or other numerical problems.
"""